# Лекција 05 - Агентски RAG


## Подешавање

Овај нотебук демонстрира Agentic RAG (Retrieval-Augmented Generation) образац користећи Microsoft Agent Framework.

**Претпоставке:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ваш крајњи ендпоинт Azure AI Search сервиса
- `AZURE_SEARCH_API_KEY` — ваш Azure AI Search API кључ
- Azure OpenAI распоред конфигурисан преко променљивих окружења
- Azure CLI аутентификован (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је Agentic RAG?

Традиционални RAG прати фиксни ток: преузми документа, затим генериши одговор. **Agentic RAG** иде даље тако што даје агенту аутономију да одлучи **када** и **како** да преузме информације.

Са Agentic RAG, агент може:
- **Одлучити** да ли је потребно преузимање пре него што одговори на питање
- **Изабрати** који извор података или алат ће користити за упит
- **Проценити** преузете резултате и извршити додатна преузимања ако први покушај није довољан
- **Комбиновати** информације из више корака преузимања у кохерентан одговор

Ово чини агента флексибилнијим и прецизнијим у поређењу са статичним током преузми-затим-генериши.


## Креирање алата за претрагу

У Agentic RAG-у, спољни извори података се обавијају као **алати** које агент може позвати по потреби. Ово омогућава агенту да третира преузимање као још једну радњу коју може извршити, уместо као обавезни корак.

Испод дефинишемо базу знања о путовањима и излажемо је као алат који агент може позвати да би пронашао информације о дестинацији.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Креирање РАГ агента

Сада правимо агента који је упућен да **увек пре одговора преузме информације**. Агент користи алат `search_travel_knowledge` да својим одговорима обезбеди основу у бази знања уместо да се ослања на своје податке о обуци.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративно проналажење — Образац правио-прегледач

Кључна предност Agentic RAG је **итеративно проналажење**. агент може извршити више рунди претраге како би верификовао, прецизирао или проширио своја почетна сазнања — слично радном току „правио-прегледач“:

1. **Правио корак**: агент проналази почетне информације и саставља одговор.
2. **Прегледач корак**: агент врши додатна проналажења да би верификовао детаље или попунио празнине.

Испод, агенту је постављено питање које захтева поређење више дестинација, што га подстиче да претражује неколико пута.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Сажетак

У овој лекцији сте научили како да изградите **Agentic RAG** систем користећи Microsoft Agent Framework:

- **Agentic RAG** омогућава агентима да аутономно одлучују када да преузму информације, чинећи преузимање динамичким, а не фиксним.
- **Алати као извори података**: Спољне базе знања (као што је Azure AI Search) обавијене су као алати које агент може да позове.
- **Итеративно преузимање**: Прилика maker-checker омогућава агенту да изведе више рунди преузимања — претражује, верификује и усавршава — пре него што произведе коначни одговор.

У продукцији, заменили бисте у меморији `TRAVEL_KNOWLEDGE_BASE` стварним Azure AI Search индексом за руковање преузимањем путних докумената у великом обиму.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање одговорности**:  
Овај документ је преведен помоћу АИ сервиса за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде што прецизнији, имајте у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешне тумачења која произлазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
